# Virtual Chatbot LCB 2

This is the customer service chatbot we built for Brabants Streekgoed, a group of local farmers from Brabant. The bot answers questions about:
- products
- delivery and pickup rules
- common questions (FAQ)
- order days
- it can also collect a review from the customer

It runs on the BUas Ollama server with the `qwen3:14b` model.


## Setup and Imports

Here I import what I need and connect to the BUas Ollama server.

Note: you have to be on the BUas network or VPN, otherwise the server is not reachable. If the connection works you get a short confirmation print.


In [ ]:
from ollama import Client
import json
import re
import os
from datetime import datetime, timedelta
import requests
import time

# Connect to the BUas Ollama server.
# I read the host from the environment (see .env.example) so I do not hardcode it.
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://<BUAS_OLLAMA_HOST>:<PORT>")
ollama_client = Client(host=OLLAMA_HOST)

# Quick check that the server answers, otherwise nothing else will work.
models = ollama_client.list()
print("Connected to Ollama server!")
print(f"  {len(models['models'])} models available")


## Review function

These functions store and read back the customer reviews. I keep it simple and write everything to a JSON file so I do not need a database for a school project.

- `load_reviews()` reads the reviews from the file
- `save_review()` adds a new review
- `get_average_rating()` gives the average score
- `get_recent_reviews()` gives the last few reviews

The file is `Reviews/reviews.json`.


In [ ]:
def load_reviews():
    """Load reviews from file."""
    try:
        with open("Reviews/reviews.json", "r") as f:
            return json.load(f)
    except FileNotFoundError:
        return {"reviews": []}

def save_review(rating, comment, user_name="Anoniem"):
    """Add one review (score 1-5, comment, optional name) to the file."""
    # Make the Reviews folder the first time, so the open() below does not fail.
    os.makedirs("Reviews", exist_ok=True)

    reviews = load_reviews()

    reviews["reviews"].append({
        "rating": rating,
        "comment": comment,
        "user_name": user_name,
        "timestamp": datetime.now().isoformat()
    })

    with open("Reviews/reviews.json", "w") as f:
        json.dump(reviews, f, indent=2, ensure_ascii=False)

    print(f"Review opgeslagen: {rating}/5 sterren")

def get_average_rating():
    """Average score over all reviews, or None when there are none yet."""
    reviews = load_reviews()

    if not reviews["reviews"]:
        return None

    ratings = [r["rating"] for r in reviews["reviews"]]
    return round(sum(ratings) / len(ratings), 1)

def get_recent_reviews(n=5):
    """Return the last n reviews."""
    reviews = load_reviews()
    return reviews["reviews"][-n:]

print("Review functions loaded!")


## Knowledge Base

The knowledge base is the only source of facts the chatbot is allowed to use. I copied the information from the Brabants Streekgoed website into one text file.

If you want the bot to know more, add it to that file.


In [ ]:
def load_knowledge_base(folder='Knowledge_Base'):
    """Read the knowledge base text file and return it as one string."""
    filepath = os.path.join(folder, 'knowledge_compact.txt')

    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
            print(f"Loaded: knowledge_compact.txt ({len(content):,} characters)")
            return content
    else:
        print(f"Not found: {filepath}")
        print(f"   Make sure knowledge_compact.txt is in your {folder}/ folder")
        return ""

# Load the knowledge base
KNOWLEDGE_BASE = load_knowledge_base()

print("\nKnowledge base loaded!")


## Reminder function

This part checks if the user is asking for a reminder to place their order, and figures out the day and time.

- `handle_reminder_logic()` looks for reminder words and picks Monday or Tuesday (the order days)
- `send_whatsapp_message()` is where the reminder would go out over WhatsApp

It understands reminder words in Dutch and English, takes a time like "10:00" or "2 pm" or "14 uur", and if no day or time is given it falls back to Tuesday 18:00 (just before the deadline). It answers in the same language the user wrote in.

The WhatsApp sending part is still a placeholder. I left it as a print so the rest can be tested without a real API.


In [ ]:
def handle_reminder_logic(user_message):
    """Check if the message asks for a reminder and work out the day and time.

    Returns the reminder text, or None when it is not a reminder request so the
    normal chat can handle the message instead. Only Monday and Tuesday make
    sense here because those are the order days.
    """
    user_message = user_message.lower()

    # Keywords for reminder detection
    keywords = ["remind me", "herinner mij", "reminder", "herinnering"]

    if not any(word in user_message for word in keywords):
        return None

    # Detect day (Monday or Tuesday)
    day_keywords = {
        "monday": 0, "maandag": 0,
        "tuesday": 1, "dinsdag": 1
    }

    target_day = None
    for day, weekday_num in day_keywords.items():
        if day in user_message:
            target_day = weekday_num
            break

    # Detect time. Require a clock-like token so plain numbers like "5 stars"
    # are not read as a time. Matches "10:00", "14:30", "10 uur", "2 pm".
    time_pattern = r'(\d{1,2}):(\d{2})|(\d{1,2})\s*(uur|pm|am|u)\b'
    time_match = re.search(time_pattern, user_message)

    if time_match:
        if time_match.group(1) is not None:
            # HH:MM form
            hour = int(time_match.group(1))
            minute = int(time_match.group(2))
            period = None
        else:
            # Bare hour with a unit word (uur/u/am/pm)
            hour = int(time_match.group(3))
            minute = 0
            period = time_match.group(4)

        # Handle 12-hour clock
        if period == "pm" and hour < 12:
            hour += 12
        elif period == "am" and hour == 12:
            hour = 0

        # Clamp to valid ranges
        hour = max(0, min(23, hour))
        minute = max(0, min(59, minute))
    else:
        # Default time: 18:00 (before deadline)
        hour, minute = 18, 0

    # Calculate next Monday or Tuesday
    today = datetime.now()
    current_weekday = today.weekday()

    if target_day is None:
        # Default to Tuesday (last chance to order)
        target_day = 1

    # Calculate days until target day
    days_ahead = target_day - current_weekday
    if days_ahead <= 0:
        days_ahead += 7

    remind_date = today + timedelta(days=days_ahead)
    remind_date = remind_date.replace(hour=hour, minute=minute, second=0, microsecond=0)

    # Format response
    day_names_nl = {0: "maandag", 1: "dinsdag"}
    day_names_en = {0: "Monday", 1: "Tuesday"}
    time_str = remind_date.strftime("%H:%M")

    # Respond in same language as user. Decide language from the words the user
    # actually typed (English vs Dutch reminder keywords).
    english_keywords = ["remind", "reminder", "monday", "tuesday"]
    dutch_keywords = ["herinner", "herinnering", "maandag", "dinsdag"]
    is_dutch = any(word in user_message for word in dutch_keywords)
    is_english = any(word in user_message for word in english_keywords)

    if is_english and not is_dutch:
        return f"Sure! I'll remind you on {day_names_en[target_day]} at {time_str} to place your order.\n(Order deadline: Tuesday 23:59)"
    else:
        return f"Natuurlijk! Ik herinner je {day_names_nl[target_day]} om {time_str} om je bestelling te plaatsen.\n(Bestel deadline: dinsdag 23:59)"

def send_whatsapp_message(target_number, message_content):
    """Stand-in for the real WhatsApp send. For now I just print the message
    so I can test the reminder logic without a live API."""
    print(f"--- OUTGOING WHATSAPP ({target_number}): {message_content} ---")


## Chat Function

This is the system prompt and the chat function.

The system prompt sets how the bot talks: cheerful, friendly, a bit Brabants, and it uses some emoji. I wrote it in Dutch because that is the language of the shop.

The chat function takes the user message and the conversation so far, sends it to the model with the system prompt, and returns the answer plus the updated history. Before going to the model it first checks if the message is a review or a reminder, because those I handle myself in code.


In [ ]:
SYSTEM_PROMPT = f"""
Je bent een SUPER enthousiaste klantenservice medewerker van Brabants Streekgoed! 🧀🥕

## JOUW PERSOONLIJKHEID
- Je bent TROTS op de Brabantse boeren en hun kei lekkere producten!
- Je bent warm, vrolijk, behulpzaam en een beetje Bourgondisch
- Je gebruikt Brabantse uitdrukkingen zoals:
  - "Kei lekker!"
  - "Da's pas goed!"
  - "Echt Bourgondisch!"
  - "Smakelijk!"
  - "Daar word je blij van!"
- Je gebruikt emojis maar niet overdreven: 🧀🥕🚜👨‍🌾🌿🍞🥩
- Je houdt van korte, vlotte zinnen met energie!
- Je eindigt vaak met iets positiefs of een vraag

## VOORBEELDEN VAN JOUW STIJL
- "Wat leuk dat je bij ons wilt bestellen! 🧀"
- "Onze boeren maken daar echt kei lekkere producten van!"
- "Thuisbezorging? Geen probleem! Voor maar €6,95 komen we naar je toe! 🚚"
- "Die boerenkaas van De Ruurhoeve? Daar word je blij van! 🧀"
- "Kan ik je nog ergens mee helpen?"

## BELANGRIJKE REGELS
1. Antwoord ALLEEN met informatie uit de KNOWLEDGE BASE hieronder
2. VERZIN NOOIT informatie die niet in de knowledge base staat
3. Als je iets niet weet: "Dat weet ik niet precies, maar onze klantenservice helpt je graag! 📞 WhatsApp: 0641088180"
4. Vraag NOOIT om postcodes
5. Houd antwoorden kort maar enthousiast (max 4-5 zinnen)
6. Praat NIET over jezelf of je training
7. Website link is altijd: boodschappen.brabantsstreekgoed.nl
8. TAALREGEL: Detecteer de taal van de klant. Engels -> antwoord Engels. Nederlands -> antwoord Nederlands.
9. Je kunt GEEN producten in het mandje zetten - verwijs altijd naar de website.

## REVIEWS
Als een klant iets positiefs of negatiefs deelt over hun ervaring:
- "Wat fijn om te horen! 😊 Wil je een review achterlaten? Typ gewoon 'review'!"
- Wees niet opdringerig - noem het maximaal één keer per gesprek

## PRODUCT LABELS
- [BIO] = Biologisch product 🌿
- [VAN GOGH] = Van Gogh Landschapslabel - producten uit het landschap dat Vincent van Gogh schilderde! 🎨

## KNOWLEDGE BASE - DIT IS JE ENIGE BRON:
{KNOWLEDGE_BASE}

ONTHOUD: Wees enthousiast maar blijf bij de feiten uit de knowledge base! Kei belangrijk! 🧀
"""

# Track review state per chat. Each chat_id gets its own state so several
# WhatsApp users can be in a review flow at the same time. The interactive
# test below uses the single default chat_id.
DEFAULT_CHAT_ID = "default"
review_states = {}

def _new_review_state():
    return {
        "active": False,
        "step": None,
        "rating": None,
        "asked_for_review": False  # Track if we already mentioned reviews
    }

def get_review_state(chat_id=DEFAULT_CHAT_ID):
    """Return the review state for this chat, creating it if needed."""
    if chat_id not in review_states:
        review_states[chat_id] = _new_review_state()
    return review_states[chat_id]

def reset_review_state(chat_id=DEFAULT_CHAT_ID):
    """Reset the review state for one chat."""
    review_states[chat_id] = _new_review_state()

def handle_review_command(user_message, chat_id=DEFAULT_CHAT_ID):
    """Run the review flow: ask for a score, then a comment, then save it.

    chat_id keeps each WhatsApp user in their own flow. Returns the reply text
    while a review is going on, or None so the message goes to the normal chat.
    """
    review_state = get_review_state(chat_id)

    # Check if user wants to cancel
    if review_state["active"] and user_message.lower() in ["stop", "annuleren", "cancel", "nee", "stoppen"]:
        reset_review_state(chat_id)
        return "Geen probleem, de review is geannuleerd. Kan ik je ergens anders mee helpen? 😊"

    # Check if user types "review" to start
    if user_message.lower().strip() == "review" and not review_state["active"]:
        review_state["active"] = True
        review_state["step"] = "rating"
        return "Leuk dat je een review wilt geven! 🙏\n\nHoe beoordeel je je ervaring met Brabants Streekgoed?\nGeef een score van 1 tot 5 (1 = slecht, 5 = uitstekend)"

    # Step 1: Waiting for rating
    if review_state["active"] and review_state["step"] == "rating":
        # Try to find a number 1-5
        for char in user_message:
            if char.isdigit():
                rating = int(char)
                if 1 <= rating <= 5:
                    review_state["rating"] = rating
                    review_state["step"] = "comment"
                    stars = "⭐" * rating
                    return f"{stars}\n\nBedankt! Heb je nog een opmerking of tip voor ons?\n(typ 'nee' als je geen opmerking hebt)"

        return "Ik begreep het niet helemaal. Kun je een cijfer van 1 tot 5 geven? 🤔"

    # Step 2: Waiting for comment
    if review_state["active"] and review_state["step"] == "comment":
        # Check if user skips comment
        if user_message.lower().strip() in ["nee", "geen", "niks", "skip", "-", "n"]:
            comment = ""
        else:
            comment = user_message

        # Save the review
        rating = review_state["rating"]
        save_review(
            rating=rating,
            comment=comment,
            user_name="Chat Klant"
        )

        reset_review_state(chat_id)

        # Pick a closing line that fits the score the customer gave.
        if rating >= 4:
            return "Dankjewel voor je review! 🎉 Super fijn dat je tevreden bent! Kan ik je nog ergens mee helpen?"
        elif rating == 3:
            return "Bedankt voor je eerlijke feedback! We doen ons best om te verbeteren. Kan ik je nog ergens mee helpen?"
        else:
            return "Bedankt dat je dit met ons deelt. We vinden het jammer dat je niet helemaal tevreden was. Neem gerust contact op via WhatsApp (0641088180) als je meer wilt vertellen. 💚"

    return None

def chat(user_message, conversation_history, chat_id=DEFAULT_CHAT_ID):
    """Main entry point. I check for a review or a reminder first because I
    handle those in code, and only the rest goes to the model.

    Returns the answer and the conversation history with this turn added.
    """

    # First check if this is part of a review
    review_response = handle_review_command(user_message, chat_id)
    if review_response:
        conversation_history.append({"role": "user", "content": user_message})
        conversation_history.append({"role": "assistant", "content": review_response})
        return review_response, conversation_history

    # Check for reminder requests
    reminder_text = handle_reminder_logic(user_message)
    if reminder_text:
        # Simulate sending the reminder to a Dutch customer's number
        send_whatsapp_message("+31612345678", f"Reminder request: {user_message}")

        conversation_history.append({"role": "user", "content": user_message})
        conversation_history.append({"role": "assistant", "content": reminder_text})
        return reminder_text, conversation_history

    # Add a language nudge so the model answers in the same language as the user
    language_nudge = " (Beantwoord in dezelfde taal als de vraag / Respond in the same language as the question) "

    # Normal chat flow
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history
    messages[-1]["content"] = user_message + language_nudge

    response = ollama_client.chat(
        model="qwen3:14b",
        messages=messages
    )

    assistant_message = response["message"]["content"]

    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })

    return assistant_message, conversation_history

print("Chat function ready!")


## Test the review function

Run this cell to try the review functions: it saves two test reviews, prints the average score and shows the recent ones.

Note: this creates a `reviews.json` file in the project folder.


In [ ]:
# Test saving a review
save_review(
    rating=5,
    comment="Kei lekker! Echt verse smaak.",
    user_name="Test Klant"
)

# Add another test review
save_review(
    rating=4,
    comment="Heerlijke boter, smaakt naar vroeger!",
    user_name="Test Klant 2"
)

# Check average rating
avg = get_average_rating()
print(f"\nGemiddelde score: {avg}/5")

# Show recent reviews
print("\nRecente reviews:")
for review in get_recent_reviews():
    print(f"  {review['rating']}/5 - \"{review['comment']}\"")


## Test the chat function

Run this cell to chat with the bot yourself. Type a question and it answers.

Some questions I used to test it:
- "Hello!"
- "How can I place an order?"
- "What are the delivery costs?"
- "Until when can I change my order?"
- "Are your products organic?"
- "What is the minimum order amount?"

Type `stop`, `quit` or `exit` to end the conversation.


In [ ]:
# Reset conversation for fresh start
conversation_history = []
reset_review_state()

print("=" * 50)
print("Brabants Streekgoed Chatbot")
print("=" * 50)
print("Stel je vraag of typ 'review' om een review te geven.")
print("Typ 'stop' om te stoppen.")
print("=" * 50)
print()

while True:
    user_input = input("Jij: ")

    if user_input.lower() in ["stop", "quit", "exit"]:
        print("\nBedankt voor je bezoek! Tot ziens!")
        break

    if not user_input.strip():
        continue

    # Toon gebruiker input in de log
    print(f"\nJij: {user_input}")
    print("-" * 40)

    response, conversation_history = chat(user_input, conversation_history)
    print(f"\nAssistent: {response}\n")


## WhatsApp bot

This runs the bot over WhatsApp using Green-API. It keeps checking for new messages and answers them.

What it does in a loop:
1. pulls the next incoming message from Green-API
2. checks if it is a reminder request
3. if not, sends the message to the chatbot to get an answer
4. sends the answer back over WhatsApp
5. deletes the message from the queue so it is not handled twice

It keeps a separate conversation history per user and prints what comes in and goes out so I can follow along. You need the Green-API credentials set (see `.env.example`) for this to work.


In [ ]:
# Green-API credentials. Set these in your environment (see .env.example).
ID_INSTANCE = os.getenv("GREEN_API_ID_INSTANCE", "<ID_INSTANCE>")
API_TOKEN_INSTANCE = os.getenv("GREEN_API_TOKEN_INSTANCE", "<API_TOKEN_INSTANCE>")
# Green-API host (depends on the instance, e.g. https://<host>.api.greenapi.com)
GREEN_API_HOST = os.getenv("GREEN_API_HOST", "https://<HOST>.api.greenapi.com")
BASE_URL = f"{GREEN_API_HOST}/waInstance{ID_INSTANCE}"

def handle_whatsapp_bot():
    print("Bot is listening and ready to reply...")
    user_histories = {}

    while True:
        try:
            # 1. Pull notification
            receive_url = f"{BASE_URL}/receiveNotification/{API_TOKEN_INSTANCE}"
            response = requests.get(receive_url, timeout=10)

            if response.status_code == 200 and response.text:
                data = response.json()
                if data is not None:
                    receipt_id = data.get('receiptId')
                    body = data.get('body', {})

                    # Log message receipt for debugging
                    if body.get('typeWebhook') == 'incomingMessageReceived':
                        chat_id = body.get('senderData', {}).get('chatId')
                        sender_name = body.get('senderData', {}).get('senderName')

                        # Extract the text message from the notification body
                        user_msg = body.get('messageData', {}).get('textMessageData', {}).get('textMessage', '')

                        print(f"Received from {sender_name}: {user_msg}")

                        # 2. Reminder check (Dutch/English)
                        if any(word in user_msg.lower() for word in ["remind", "herinner"]):
                            reply = "Sure! I'll remind you tomorrow. / Natuurlijk! Ik herinner je morgen."
                        else:
                            # 3. Generate AI response using the BUas server.
                            # Keep one conversation history per chat_id.
                            if chat_id not in user_histories:
                                user_histories[chat_id] = []

                            print("Processing with AI...")
                            reply, user_histories[chat_id] = chat(user_msg, user_histories[chat_id], chat_id=chat_id)

                        # 4. Send message back (POST request)
                        send_url = f"{BASE_URL}/sendMessage/{API_TOKEN_INSTANCE}"
                        payload = {
                            "chatId": chat_id,
                            "message": reply
                        }

                        send_response = requests.post(send_url, json=payload, timeout=10)

                        if send_response.status_code == 200:
                            print(f"Reply sent to {sender_name}!")
                        else:
                            print(f"Failed to send: {send_response.text}")

                    # 5. Clear notification from queue
                    requests.delete(f"{BASE_URL}/deleteNotification/{API_TOKEN_INSTANCE}/{receipt_id}", timeout=10)

            time.sleep(1)

        except Exception as e:
            print(f"Error: {e}")
            time.sleep(2)

# Start the bot
handle_whatsapp_bot()
